In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od

In [ ]:
url = "https://www.kaggle.com/c/new-york-city-taxi-fare-prediction"

In [ ]:
od.download(url)

Skipping, found downloaded files in "./new-york-city-taxi-fare-prediction" (use force=True to force download)


In [ ]:
import pandas as pd

In [ ]:
selected_cols = 'fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count'.split(',')

In [ ]:
dtypes = {
    'fare_amount': 'float32',
    'pickup_datetime' : 'float32',
    'pickup_longitude': 'float32',
    'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32',
    'dropoff_latitude': 'float32',
    'passengercount' : 'uint8',
}

In [ ]:
sample_fraction = 0.01

In [ ]:
import random

In [ ]:
random.random()

0.18517966836690347

In [ ]:
data_dir = "./new-york-city-taxi-fare-prediction"

In [ ]:
def skip_row(row_idx):
    if row_idx == 0:
        return False
    return random.random() > sample_fraction

random.seed(42)
df = pd.read_csv(
    data_dir + '/train.csv',
    usecols=selected_cols,
    parse_dates=['pickup_datetime'],
    dtype=dtypes,
    skiprows=skip_row
)

In [ ]:
test_df = pd.read_csv(
    data_dir + '/test.csv',
    dtype=dtypes,
    parse_dates=['pickup_datetime']
)

EDA

In [ ]:
eda_df = df.copy()

Values ranges

In [ ]:
eda_df = eda_df[
    (eda_df['pickup_longitude'].between(-75, -72)) &
    (eda_df['dropoff_longitude'].between(-75, -72)) &
    (eda_df['pickup_latitude'].between(40, 42)) &
    (eda_df['dropoff_latitude'].between(40, 42))
]

eda_df = eda_df[
    (eda_df['passenger_count'] >= 1) &
    (eda_df['passenger_count'] <= 6)
]

In [ ]:
eda_df = eda_df[
    (eda_df['fare_amount'] > 0) &
    (eda_df['fare_amount'] < 500)
]

In [ ]:
eda_df.describe()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,538849.000000,538849.000000,538849.000000,538849.000000,538849.000000,538849.000000
mean,11.343322,-73.975128,40.750988,-73.974365,40.751347,1.690819
std,9.725473,0.211196,0.084345,0.210987,0.085252,1.306897
min,0.010000,-74.934593,40.063896,-74.946442,40.054207,1.000000
25%,6.000000,-73.992241,40.736542,-73.991608,40.735527,1.000000
50%,8.500000,-73.982101,40.753338,-73.980621,40.753796,1.000000
75%,12.500000,-73.968376,40.767464,-73.965416,40.768372,2.000000
max,499.000000,-72.471581,41.787712,-72.113823,41.806301,6.000000


In [ ]:
test_df.describe(
)

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,9914.000000,9914.000000,9914.000000,9914.000000,9914.000000
mean,-73.974716,40.751041,-73.973656,40.751740,1.671273
std,0.042799,0.033542,0.039093,0.035436,1.278747
min,-74.252190,40.573143,-74.263245,40.568974,1.000000
25%,-73.992500,40.736125,-73.991249,40.735253,1.000000
50%,-73.982327,40.753052,-73.980015,40.754065,1.000000
75%,-73.968012,40.767113,-73.964062,40.768757,2.000000
max,-72.986534,41.709557,-72.990967,41.696682,6.000000


Spliting date time

In [ ]:
def add_dateparts(df, col):
    df[col + '_year'] = df[col].dt.year
    df[col + '_month'] = df[col].dt.month
    df[col + '_day'] = df[col].dt.day
    df[col + '_weekday'] = df[col].dt.weekday
    df[col + '_hour'] = df[col].dt.hour

In [ ]:
add_dateparts(eda_df, 'pickup_datetime')

In [ ]:
add_dateparts(test_df, 'pickup_datetime')

In [ ]:
eda_df

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_datetime_year,pickup_datetime_month,pickup_datetime_day,pickup_datetime_weekday,pickup_datetime_hour
0,4.0,2014-12-06 20:36:22+00:00,-73.979813,40.751904,-73.979446,40.755482,1,2014,12,6,5,20
2,8.9,2011-06-15 18:07:00+00:00,-73.996330,40.753223,-73.978897,40.766964,3,2011,6,15,2,18
3,6.9,2009-12-14 12:33:00+00:00,-73.982430,40.745747,-73.982430,40.745747,1,2009,12,14,0,12
4,7.0,2013-11-06 11:26:54+00:00,-73.959061,40.781059,-73.962059,40.768604,1,2013,11,6,2,11
5,15.5,2014-12-08 01:00:16+00:00,-73.957672,40.717888,-73.942581,40.686398,1,2014,12,8,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
552445,45.0,2014-02-06 23:59:45+00:00,-73.973587,40.747669,-73.999916,40.602894,1,2014,2,6,3,23
552446,22.5,2015-01-05 15:29:08+00:00,-73.935928,40.799656,-73.985710,40.726952,2,2015,1,5,0,15
552447,4.5,2013-02-17 22:27:00+00:00,-73.992531,40.748619,-73.998436,40.740143,1,2013,2,17,6,22
552448,14.5,2013-01-27 12:41:00+00:00,-74.012115,40.706635,-73.988724,40.756218,1,2013,1,27,6,12


Column for distance between coordinates

In [ ]:
import numpy as np

In [ ]:
# lets add a column which specifies the distance using haversine formula
def haversine_np(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)

    All args must be of equal length.
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))

    km = 6367 * c
    return km

In [ ]:
def add_trip_distance(df):
    df['trip_distance'] = haversine_np(
        df['pickup_longitude'],
        df['pickup_latitude'],
        df['dropoff_longitude'],
        df['dropoff_latitude']
    )

In [ ]:
add_trip_distance(eda_df)

In [ ]:
add_trip_distance(test_df)

In [ ]:
eda_df.columns

Index(['fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count',
       'pickup_datetime_year', 'pickup_datetime_month', 'pickup_datetime_day',
       'pickup_datetime_weekday', 'pickup_datetime_hour', 'trip_distance'],
      dtype='object')

In [ ]:
eda_df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_datetime_year,pickup_datetime_month,pickup_datetime_day,pickup_datetime_weekday,pickup_datetime_hour,trip_distance
0,4.0,2014-12-06 20:36:22+00:00,-73.979813,40.751904,-73.979446,40.755482,1,2014,12,6,5,20,0.398929
2,8.9,2011-06-15 18:07:00+00:00,-73.996330,40.753223,-73.978897,40.766964,3,2011,6,15,2,18,2.117704
3,6.9,2009-12-14 12:33:00+00:00,-73.982430,40.745747,-73.982430,40.745747,1,2009,12,14,0,12,0.000000
4,7.0,2013-11-06 11:26:54+00:00,-73.959061,40.781059,-73.962059,40.768604,1,2013,11,6,2,11,1.406860
5,15.5,2014-12-08 01:00:16+00:00,-73.957672,40.717888,-73.942581,40.686398,1,2014,12,8,0,1,3.722735


In [ ]:
eda_df = eda_df[eda_df['trip_distance'] != 0]

In [ ]:
eda_df

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_datetime_year,pickup_datetime_month,pickup_datetime_day,pickup_datetime_weekday,pickup_datetime_hour,trip_distance
0,4.0,2014-12-06 20:36:22+00:00,-73.979813,40.751904,-73.979446,40.755482,1,2014,12,6,5,20,0.398929
2,8.9,2011-06-15 18:07:00+00:00,-73.996330,40.753223,-73.978897,40.766964,3,2011,6,15,2,18,2.117704
4,7.0,2013-11-06 11:26:54+00:00,-73.959061,40.781059,-73.962059,40.768604,1,2013,11,6,2,11,1.406860
5,15.5,2014-12-08 01:00:16+00:00,-73.957672,40.717888,-73.942581,40.686398,1,2014,12,8,0,1,3.722735
6,6.0,2012-09-26 13:14:47+00:00,-73.964485,40.764431,-73.956573,40.779854,1,2012,9,26,2,13,1.838763
...,...,...,...,...,...,...,...,...,...,...,...,...,...
552445,45.0,2014-02-06 23:59:45+00:00,-73.973587,40.747669,-73.999916,40.602894,1,2014,2,6,3,23,16.240583
552446,22.5,2015-01-05 15:29:08+00:00,-73.935928,40.799656,-73.985710,40.726952,2,2015,1,5,0,15,9.100983
552447,4.5,2013-02-17 22:27:00+00:00,-73.992531,40.748619,-73.998436,40.740143,1,2013,2,17,6,22,1.064929
552448,14.5,2013-01-27 12:41:00+00:00,-74.012115,40.706635,-73.988724,40.756218,1,2013,1,27,6,12,5.851552


ading distance from popular landmarks

In [ ]:
jfk_lonlat = -73.7781, 40.6413
lga_lonlat = -73.8740, 40.7769
ewr_lonlat = -74.1745, 40.6895
met_lonlat = -73.9632, 40.7794
wtc_lonlat = -74.0099, 40.7126


def add_landmark_dropoff_distance(df, landmark_name, landmark_lonlat):
    lon, lat = landmark_lonlat
    df[landmark_name + '_drop_distance'] = haversine_np(
        lon, lat,
        df['dropoff_longitude'],
        df['dropoff_latitude']
    )

In [ ]:
def add_landmarks(df):
    landmarks = [
        ('jfk', jfk_lonlat),
        ('lga', lga_lonlat),
        ('ewr', ewr_lonlat),
        ('met', met_lonlat),
        ('wtc', wtc_lonlat)
    ]

    for name, lonlat in landmarks:
        add_landmark_dropoff_distance(df, name, lonlat)

In [ ]:
add_landmarks(eda_df)

/tmp/ipykernel_16243/1812775617.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[landmark_name + '_drop_distance'] = haversine_np(
/tmp/ipykernel_16243/1812775617.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[landmark_name + '_drop_distance'] = haversine_np(
/tmp/ipykernel_16243/1812775617.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

In [ ]:
add_landmarks(test_df)

In [ ]:
eda_df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_datetime_year,pickup_datetime_month,pickup_datetime_day,pickup_datetime_weekday,pickup_datetime_hour,trip_distance,jfk_drop_distance,lga_drop_distance,ewr_drop_distance,met_drop_distance,wtc_drop_distance
0,4.0,2014-12-06 20:36:22+00:00,-73.979813,40.751904,-73.979446,40.755482,1,2014,12,6,5,20,0.398929,21.183979,9.188184,17.989639,2.988609,5.411651
2,8.9,2011-06-15 18:07:00+00:00,-73.996330,40.753223,-73.978897,40.766964,3,2011,6,15,2,18,2.117704,21.935322,8.896636,18.585823,1.911547,6.581114
4,7.0,2013-11-06 11:26:54+00:00,-73.959061,40.781059,-73.962059,40.768604,1,2013,11,6,2,11,1.406860,20.982895,7.467426,19.933247,1.203489,7.413479
5,15.5,2014-12-08 01:00:16+00:00,-73.957672,40.717888,-73.942581,40.686398,1,2014,12,8,0,1,3.722735,14.742958,11.597059,19.545231,10.479470,6.375153
6,6.0,2012-09-26 13:14:47+00:00,-73.964485,40.764431,-73.956573,40.779854,1,2012,9,26,2,13,1.838763,21.519035,6.955910,20.917858,0.560350,8.718511


Random cleaning

In [ ]:
eda_df = eda_df.drop(columns=['pickup_datetime'])

In [ ]:
test_df = test_df.drop(columns=['pickup_datetime'])

In [ ]:
eda_df.columns

Index(['fare_amount', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count',
       'pickup_datetime_year', 'pickup_datetime_month', 'pickup_datetime_day',
       'pickup_datetime_weekday', 'pickup_datetime_hour', 'trip_distance',
       'jfk_drop_distance', 'lga_drop_distance', 'ewr_drop_distance',
       'met_drop_distance', 'wtc_drop_distance'],
      dtype='object')

In [ ]:
eda_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 532969 entries, 0 to 552449
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   fare_amount              532969 non-null  float32
 1   pickup_longitude         532969 non-null  float32
 2   pickup_latitude          532969 non-null  float32
 3   dropoff_longitude        532969 non-null  float32
 4   dropoff_latitude         532969 non-null  float32
 5   passenger_count          532969 non-null  int64  
 6   pickup_datetime_year     532969 non-null  int32  
 7   pickup_datetime_month    532969 non-null  int32  
 8   pickup_datetime_day      532969 non-null  int32  
 9   pickup_datetime_weekday  532969 non-null  int32  
 10  pickup_datetime_hour     532969 non-null  int32  
 11  trip_distance            532969 non-null  float32
 12  jfk_drop_distance        532969 non-null  float32
 13  lga_drop_distance        532969 non-null  float32
 14  ewr_drop_

In [ ]:
eda_df

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_datetime_year,pickup_datetime_month,pickup_datetime_day,pickup_datetime_weekday,pickup_datetime_hour,trip_distance,jfk_drop_distance,lga_drop_distance,ewr_drop_distance,met_drop_distance,wtc_drop_distance
0,4.0,-73.979813,40.751904,-73.979446,40.755482,1,2014,12,6,5,20,0.398929,21.183979,9.188184,17.989639,2.988609,5.411651
2,8.9,-73.996330,40.753223,-73.978897,40.766964,3,2011,6,15,2,18,2.117704,21.935322,8.896636,18.585823,1.911547,6.581114
4,7.0,-73.959061,40.781059,-73.962059,40.768604,1,2013,11,6,2,11,1.406860,20.982895,7.467426,19.933247,1.203489,7.413479
5,15.5,-73.957672,40.717888,-73.942581,40.686398,1,2014,12,8,0,1,3.722735,14.742958,11.597059,19.545231,10.479470,6.375153
6,6.0,-73.964485,40.764431,-73.956573,40.779854,1,2012,9,26,2,13,1.838763,21.519035,6.955910,20.917858,0.560350,8.718511
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552445,45.0,-73.973587,40.747669,-73.999916,40.602894,1,2014,2,6,3,23,16.240583,19.190046,22.055864,17.587349,19.856646,12.220176
552446,22.5,-73.935928,40.799656,-73.985710,40.726952,2,2015,1,5,0,15,9.100983,19.916544,10.919534,16.439034,6.128220,2.587632
552447,4.5,-73.992531,40.748619,-73.998436,40.740143,1,2013,2,17,6,22,1.064929,21.571228,11.242204,15.861959,5.274842,3.209442
552448,14.5,-74.012115,40.706635,-73.988724,40.756218,1,2013,1,27,6,12,5.851552,21.862238,9.924991,17.313942,3.353854,5.164717


Spliting data

In [ ]:
input_cols = 'pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,trip_distance,jfk_drop_distance,lga_drop_distance,ewr_drop_distance,met_drop_distance,wtc_drop_distance'.split(',')

In [ ]:
target_col = 'fare_amount'

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df,val_df = train_test_split(eda_df,test_size=0.2,random_state=42)

In [ ]:
len(train_df), len(val_df)

(426375, 106594)

In [ ]:
train_df.columns

Index(['fare_amount', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count',
       'pickup_datetime_year', 'pickup_datetime_month', 'pickup_datetime_day',
       'pickup_datetime_weekday', 'pickup_datetime_hour', 'trip_distance',
       'jfk_drop_distance', 'lga_drop_distance', 'ewr_drop_distance',
       'met_drop_distance', 'wtc_drop_distance'],
      dtype='object')

In [ ]:
train_input = train_df[input_cols]
train_target = train_df[target_col]

In [ ]:
val_input = val_df[input_cols]
val_target = val_df[target_col]

Baseline model

In [ ]:
#Baseline model
base = np.full(train_input.shape[0],train_target.mean())

In [ ]:
base

array([11.341759, 11.341759, 11.341759, ..., 11.341759, 11.341759,
       11.341759], dtype=float32)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
def rmse(y_true,y_pred):
    return np.sqrt(mean_squared_error(y_true,y_pred))

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
model = LinearRegression().fit(train_input,train_target)

In [ ]:
train_pred = model.predict(train_input)
val_pred = model.predict(val_input)

In [ ]:
print("Baseline:", rmse(train_target, base))
print("Model:", rmse(train_target, train_pred))

Baseline: 9.63018500636597
Model: 4.947254769806677


In [ ]:
val_base = np.full(val_input.shape[0], train_target.mean())

print("Val Baseline:", rmse(val_target, val_base))
print("Val Model:", rmse(val_target, val_pred))

Val Baseline: 9.604594212171005
Val Model: 4.724790807001769


Using different models

In [ ]:
def evaluate(model):
  train_pred = model.predict(train_input)
  train_rmse = rmse(train_target, train_pred)

  val_pred = model.predict(val_input)
  val_rmse = rmse(val_target, val_pred)

  return train_rmse, val_rmse

Ridge Regression


In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
model_1 = Ridge(random_state=42,alpha=0.9).fit(train_input,train_target)

In [ ]:
evaluate(model_1)

(np.float64(4.9472666084299926), np.float64(4.724778611272814))

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
#model_2 = RandomForestRegressor(random_state=42,n_estimators=100,n_jobs=-1).fit(train_input,train_target)

KeyboardInterrupt: 

XGBoost

In [ ]:
from xgboost import XGBRegressor

In [ ]:
model3 = XGBRegressor(
    max_depth=6,
    objective='reg:squarederror',
    n_estimators=250,
    random_state=42,
    n_jobs=-1,
    learning_rate=0.1
).fit(train_input,train_target)

In [ ]:
evaluate(model3)

(np.float64(3.451815810088914), np.float64(3.6591431997947286))

Submission

In [ ]:
test_df.head()

,key,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_datetime_year,pickup_datetime_month,pickup_datetime_day,pickup_datetime_weekday,pickup_datetime_hour,trip_distance,jfk_drop_distance,lga_drop_distance,ewr_drop_distance,met_drop_distance,wtc_drop_distance
0,2015-01-27 13:08:24.0000002,-73.973320,40.763805,-73.981430,40.743835,1,2015,1,27,1,13,2.321899,20.574911,9.760167,17.346842,4.239343,4.218709
1,2015-01-27 13:08:24.0000003,-73.986862,40.719383,-73.998886,40.739201,1,2015,1,27,1,13,2.423777,21.550976,11.315990,15.789623,5.382879,3.098136
2,2011-10-08 11:53:44.0000002,-73.982521,40.751259,-73.979652,40.746140,1,2011,10,8,5,11,0.618015,20.594069,9.526829,17.576965,3.946721,4.514503
3,2012-12-01 21:12:12.0000002,-73.981163,40.767807,-73.990448,40.751637,1,2012,12,1,5,21,1.959681,21.689365,10.195091,16.969650,3.843892,4.637048
4,2012-12-01 21:12:12.0000003,-73.966049,40.789776,-73.988564,40.744427,1,2012,12,1,5,21,5.383829,21.113993,10.295857,16.808367,4.433764,3.967223


In [ ]:
def predict_and_submit(model, fname):
    test_preds = model.predict(test_df[input_cols])
    sub_df = pd.read_csv(data_dir + '/sample_submission.csv')
    sub_df['fare_amount'] = test_preds
    sub_df.to_csv(fname, index=None)
    return sub_df

In [ ]:
submission_df = pd.read_csv(data_dir + '/sample_submission.csv')

In [ ]:
submission_df.head()

,key,fare_amount
0,2015-01-27 13:08:24.0000002,11.35
1,2015-01-27 13:08:24.0000003,11.35
2,2011-10-08 11:53:44.0000002,11.35
3,2012-12-01 21:12:12.0000002,11.35
4,2012-12-01 21:12:12.0000003,11.35


In [ ]:
predict_and_submit(model3,'xg_submission.csv')

,key,fare_amount
0,2015-01-27 13:08:24.0000002,8.647932
1,2015-01-27 13:08:24.0000003,9.367023
2,2011-10-08 11:53:44.0000002,5.031875
3,2012-12-01 21:12:12.0000002,8.592746
4,2012-12-01 21:12:12.0000003,14.931952
...,...,...
9909,2015-05-10 12:37:51.0000002,8.434605
9910,2015-01-12 17:05:51.0000001,11.367782
9911,2015-04-19 20:44:15.0000001,54.596050
9912,2015-01-31 01:05:19.0000005,18.506102
